# 🌾 Can AI Predict Irrigation?

Welcome! Today we are going to build a tiny machine learning model that tries to predict how much irrigation happened on a farm.

You do **not** need to know coding or machine learning already. The notebook is written like a recipe:

1. Run each code cell from top to bottom.
2. Read the short notes.
3. Change the values in the boxes/sliders when asked.
4. See how the model changes.

## The story

A farmer has a center pivot irrigation system. The field gets different weather each day: hot days, windy days, rainy days, dry days.

We will ask:

> Can a computer learn a pattern between weather and irrigation?

At the end, we will enter **today's weather** and ask the model to guess today's irrigation.

## 🧠 What is machine learning?

Machine learning is not magic. It is pattern finding.

Instead of writing a rule like:

> If it is hot and dry, irrigate more.

we give the computer examples like:

| Weather | Irrigation |
|---|---:|
| Hot and dry | More water |
| Cool and rainy | Less water |

Then the model tries to learn the pattern.

In this notebook, we will try two models:

1. **Linear Regression**: a simple formula.
2. **Small Neural Network**: a tiny AI model made of connected neurons.

## ▶️ Step 1: Load our tools

Run the cell below. It loads the Python tools we need.

Do not worry about understanding every line. Think of this as taking tools out of a toolbox.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import math
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output, Markdown
    WIDGETS_AVAILABLE = True
except Exception:
    WIDGETS_AVAILABLE = False

try:
    import tensorflow as tf
    TF_AVAILABLE = True
    tf.random.set_seed(42)
except Exception:
    TF_AVAILABLE = False

np.random.seed(42)
random.seed(42)

print("✅ Tools loaded!")
print(f"Interactive widgets available: {WIDGETS_AVAILABLE}")
print(f"TensorFlow available: {TF_AVAILABLE}")

## 🌦️ Step 2: Load the farm weather + irrigation data

This notebook looks for data in this order:

1. `data/merged_irrigation_weather.csv`
2. `../data/merged_irrigation_weather.csv`
3. `data/weather.csv`
4. `../data/weather.csv`

If we only have weather data, the notebook creates a **toy irrigation amount** so the activity still works.

When we get the farmer's real data, we will replace the toy irrigation column with the real `irrigation_amount` column.

In [ ]:
def find_data_file():
    candidates = [
        Path("data/merged_irrigation_weather.csv"),
        Path("../data/merged_irrigation_weather.csv"),
        Path("data/weather.csv"),
        Path("../data/weather.csv"),
    ]
    for path in candidates:
        if path.exists():
            return path
    return None

path = find_data_file()

if path is None:
    print("⚠️ No CSV found. Creating a tiny fake weather dataset so the notebook still runs.")
    dates = pd.date_range("2024-06-01", periods=120, freq="D")
    df = pd.DataFrame({"date": dates})
    df["max_temp_f"] = 75 + 18*np.sin(np.arange(len(df))/18) + np.random.normal(0, 5, len(df))
    df["min_temp_f"] = df["max_temp_f"] - np.random.uniform(18, 28, len(df))
    df["mean_temp_f"] = (df["max_temp_f"] + df["min_temp_f"]) / 2
    df["mean_humidity_pct"] = 55 - 0.35*(df["max_temp_f"] - 80) + np.random.normal(0, 8, len(df))
    df["precip_in"] = np.where(np.random.rand(len(df)) < 0.12, np.random.exponential(0.12, len(df)), 0)
    df["avg_wind_mph"] = np.random.uniform(3, 14, len(df))
    df["max_wind_mph"] = df["avg_wind_mph"] + np.random.uniform(3, 15, len(df))
    df["et0_in"] = 0.08 + 0.004*(df["max_temp_f"] - 70) + 0.003*df["avg_wind_mph"]
    df["source"] = "fake_demo_data"
else:
    print(f"✅ Loaded: {path}")
    df = pd.read_csv(path)
    df["source"] = "csv_file"

# Clean dates
if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"])
else:
    raise ValueError("The dataset needs a 'date' column.")

# If irrigation_amount is missing, make a toy target using weather.
# This lets students practice before real farmer data arrives.
if "irrigation_amount" not in df.columns:
    print("⚠️ No farmer irrigation column found. Creating a TOY irrigation_amount for practice.")
    for col in ["max_temp_f", "mean_humidity_pct", "precip_in", "avg_wind_mph", "et0_in"]:
        if col not in df.columns:
            df[col] = 0
    toy = (
        0.02
        + 0.010 * np.maximum(df["max_temp_f"] - 70, 0)
        + 0.80 * df["et0_in"].fillna(0)
        + 0.004 * df["avg_wind_mph"].fillna(0)
        - 0.35 * df["precip_in"].fillna(0)
        - 0.002 * df["mean_humidity_pct"].fillna(50)
        + np.random.normal(0, 0.035, len(df))
    )
    df["irrigation_amount"] = np.clip(toy, 0, None)
    df["irrigation_unit"] = "toy_inches"
else:
    if "irrigation_unit" not in df.columns:
        df["irrigation_unit"] = "unknown"

# Add engineered features if they are missing.
df = df.sort_values("date").reset_index(drop=True)

if "mean_temp_f" not in df.columns and {"max_temp_f", "min_temp_f"}.issubset(df.columns):
    df["mean_temp_f"] = (df["max_temp_f"] + df["min_temp_f"]) / 2

if "temp_3day_avg_f" not in df.columns:
    df["temp_3day_avg_f"] = df["mean_temp_f"].rolling(3, min_periods=1).mean()
if "temp_5day_avg_f" not in df.columns:
    df["temp_5day_avg_f"] = df["mean_temp_f"].rolling(5, min_periods=1).mean()
if "precip_3day_sum_in" not in df.columns:
    df["precip_3day_sum_in"] = df["precip_in"].rolling(3, min_periods=1).sum()
if "precip_5day_sum_in" not in df.columns:
    df["precip_5day_sum_in"] = df["precip_in"].rolling(5, min_periods=1).sum()
if "wind_3day_avg_mph" not in df.columns:
    df["wind_3day_avg_mph"] = df["avg_wind_mph"].rolling(3, min_periods=1).mean()
if "et0_3day_sum_in" not in df.columns:
    df["et0_3day_sum_in"] = df["et0_in"].rolling(3, min_periods=1).sum()
if "day_of_year" not in df.columns:
    df["day_of_year"] = df["date"].dt.dayofyear
if "month" not in df.columns:
    df["month"] = df["date"].dt.month

# Irrigation history features, useful once real data exists.
if "previous_irrigation_amount" not in df.columns:
    df["previous_irrigation_amount"] = df["irrigation_amount"].shift(1).fillna(0)

if "days_since_irrigation" not in df.columns:
    days = []
    last = None
    for i, value in enumerate(df["irrigation_amount"]):
        if value > 0.001:
            last = i
            days.append(0)
        elif last is None:
            days.append(np.nan)
        else:
            days.append(i - last)
    df["days_since_irrigation"] = pd.Series(days).fillna(df.index + 1)

# Keep only rows with a target.
df = df.dropna(subset=["irrigation_amount"]).reset_index(drop=True)

print(f"Rows in dataset: {len(df)}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
df.head()

## 🔍 Step 3: What do the columns mean?

A **variable** is a column in the data.

Some variables are inputs. These are clues the model can use.

The target is what the model tries to guess.

Our target is:

```text
irrigation_amount
```

That means the model is trying to answer:

> How much irrigation happened that day?

In [ ]:
variable_guide = pd.DataFrame([
    ["max_temp_f", "Highest temperature that day", "Hotter days may need more water"],
    ["mean_humidity_pct", "Average humidity", "Dry air can increase water loss"],
    ["precip_in", "Rain that day", "Rain may reduce irrigation"],
    ["avg_wind_mph", "Average wind speed", "Wind can increase evaporation"],
    ["et0_in", "Reference evapotranspiration", "Estimate of atmospheric water demand"],
    ["temp_3day_avg_f", "Average temperature over 3 days", "Captures recent heat"],
    ["precip_3day_sum_in", "Rain over 3 days", "Captures recent rainfall"],
    ["previous_irrigation_amount", "Yesterday's irrigation", "Farmers may adjust based on recent irrigation"],
    ["days_since_irrigation", "Days since last irrigation", "Helps model timing"],
    ["irrigation_amount", "Irrigation amount", "This is what we predict"],
], columns=["Variable", "What it measures", "Why it matters"])

variable_guide

## 📊 Step 4: Explore the data

Let's plot the weather and irrigation over time.

Question to ask yourself:

> Do hot/dry periods seem to line up with more irrigation?

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(df["date"], df["max_temp_f"], label="Max temperature (F)")
plt.title("Temperature over time")
plt.xlabel("Date")
plt.ylabel("Degrees F")
plt.legend()
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(df["date"], df["irrigation_amount"], label="Irrigation amount")
plt.title("Irrigation over time")
plt.xlabel("Date")
plt.ylabel("Irrigation amount")
plt.legend()
plt.show()

## 🎮 Step 5: Pick a clue and see if it relates to irrigation

Use the dropdown to choose one weather clue.

A strong clue will show a visible pattern.

In [ ]:
def scatter_feature(feature):
    plt.figure(figsize=(7, 5))
    plt.scatter(df[feature], df["irrigation_amount"], alpha=0.65)
    plt.xlabel(feature)
    plt.ylabel("irrigation_amount")
    plt.title(f"Does {feature} help predict irrigation?")
    plt.show()

feature_options = [
    c for c in [
        "max_temp_f", "mean_temp_f", "mean_humidity_pct", "precip_in",
        "avg_wind_mph", "et0_in", "temp_3day_avg_f", "precip_3day_sum_in",
        "previous_irrigation_amount", "days_since_irrigation", "day_of_year"
    ] if c in df.columns
]

if WIDGETS_AVAILABLE:
    widgets.interact(scatter_feature, feature=widgets.Dropdown(options=feature_options, value=feature_options[0]))
else:
    scatter_feature(feature_options[0])

## 🧪 Step 6: Choose the model's clues

The model can only learn from the columns we give it.

Try using a few features first. Later, add more and see if the score improves.

Good starter features:

```text
temp_3day_avg_f
precip_3day_sum_in
avg_wind_mph
mean_humidity_pct
```

In [ ]:
ALL_FEATURES = [
    "max_temp_f",
    "min_temp_f",
    "mean_temp_f",
    "mean_humidity_pct",
    "precip_in",
    "avg_wind_mph",
    "max_wind_mph",
    "et0_in",
    "temp_3day_avg_f",
    "temp_5day_avg_f",
    "precip_3day_sum_in",
    "precip_5day_sum_in",
    "wind_3day_avg_mph",
    "et0_3day_sum_in",
    "previous_irrigation_amount",
    "days_since_irrigation",
    "day_of_year",
    "month",
]

ALL_FEATURES = [c for c in ALL_FEATURES if c in df.columns]
DEFAULT_FEATURES = [c for c in ["temp_3day_avg_f", "precip_3day_sum_in", "avg_wind_mph", "mean_humidity_pct"] if c in ALL_FEATURES]

print("Features available:")
for f in ALL_FEATURES:
    print(" -", f)

## 📏 Step 7: Split the data into practice and test days

We train on some days and test on different days.

This is like studying with practice questions, then taking a quiz on questions you have not seen before.

In [ ]:
def make_train_test(feature_cols, test_size=0.25):
    clean = df[["date", "irrigation_amount"] + feature_cols].dropna().copy()
    X = clean[feature_cols]
    y = clean["irrigation_amount"]

    X_train, X_test, y_train, y_test, date_train, date_test = train_test_split(
        X, y, clean["date"], test_size=test_size, random_state=42
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    return clean, X_train, X_test, y_train, y_test, date_train, date_test, scaler, X_train_scaled, X_test_scaled

print("✅ Helper function ready.")

# 🤖 Model 1: Linear Regression

Linear regression learns a simple formula.

It is like saying:

```text
irrigation = a little bit of temperature + a little bit of wind - a little bit of rain + ...
```

It is simple, fast, and surprisingly useful.

In [ ]:
def train_linear_regression(feature_cols):
    clean, X_train, X_test, y_train, y_test, date_train, date_test, scaler, X_train_scaled, X_test_scaled = make_train_test(feature_cols)

    model = LinearRegression()
    model.fit(X_train_scaled, y_train)

    pred = model.predict(X_test_scaled)
    pred = np.clip(pred, 0, None)

    mae = mean_absolute_error(y_test, pred)
    r2 = r2_score(y_test, pred)

    results = pd.DataFrame({
        "date": date_test.values,
        "actual_irrigation": y_test.values,
        "predicted_irrigation": pred,
        "error": pred - y_test.values,
    }).sort_values("date")

    print("✅ Linear Regression trained!")
    print(f"Mean absolute error: {mae:.3f}")
    print(f"R² score: {r2:.3f}")
    print("\nMean absolute error means: on average, how far away was the model's guess?")

    plt.figure(figsize=(6, 6))
    plt.scatter(results["actual_irrigation"], results["predicted_irrigation"], alpha=0.7)
    max_val = max(results["actual_irrigation"].max(), results["predicted_irrigation"].max())
    plt.plot([0, max_val], [0, max_val], linestyle="--")
    plt.xlabel("Actual irrigation")
    plt.ylabel("Predicted irrigation")
    plt.title("Linear Regression: Actual vs Predicted")
    plt.show()

    return model, scaler, results, feature_cols

linear_model, linear_scaler, linear_results, linear_features = train_linear_regression(DEFAULT_FEATURES)
linear_results.head()

## 🎛️ Try your own Linear Regression

Pick which clues the model can use.

Then click **Train linear model**.

Things to try:

- Use only temperature.
- Use temperature + rain.
- Use rolling averages.
- Use many features.

Does more data always make the model better?

In [ ]:
if WIDGETS_AVAILABLE:
    feature_picker = widgets.SelectMultiple(
        options=ALL_FEATURES,
        value=tuple(DEFAULT_FEATURES),
        description="Features",
        rows=min(12, len(ALL_FEATURES)),
        layout=widgets.Layout(width="420px")
    )
    button = widgets.Button(description="Train linear model", button_style="success")
    output = widgets.Output()

    def on_train_linear_clicked(_):
        with output:
            clear_output()
            chosen = list(feature_picker.value)
            if len(chosen) == 0:
                print("Pick at least one feature.")
                return
            global linear_model, linear_scaler, linear_results, linear_features
            linear_model, linear_scaler, linear_results, linear_features = train_linear_regression(chosen)
            display(linear_results.head(10))

    button.on_click(on_train_linear_clicked)
    display(feature_picker, button, output)
else:
    print("Widgets are not available. Edit DEFAULT_FEATURES in the earlier cell and rerun.")

# 🧠 Model 2: A Small Neural Network

A neural network is made of small units called neurons.

Each neuron looks at the inputs, does a tiny calculation, and passes information to the next layer.

For this activity, we will use a very small neural network so it trains quickly.

You can change:

- Number of neurons
- Number of training rounds, called epochs
- Which features the model sees

In [ ]:
def train_neural_network(feature_cols, hidden_neurons=8, epochs=80, learning_rate=0.01):
    clean, X_train, X_test, y_train, y_test, date_train, date_test, scaler, X_train_scaled, X_test_scaled = make_train_test(feature_cols)

    if not TF_AVAILABLE:
        print("⚠️ TensorFlow is not installed in this environment.")
        print("Install tensorflow or run this notebook in Binder/Colab with tensorflow available.")
        return None, scaler, None, feature_cols

    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(len(feature_cols),)),
        tf.keras.layers.Dense(hidden_neurons, activation="relu"),
        tf.keras.layers.Dense(1)
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="mean_absolute_error",
        metrics=["mean_absolute_error"]
    )

    history = model.fit(
        X_train_scaled,
        y_train,
        validation_split=0.2,
        epochs=epochs,
        verbose=0
    )

    pred = model.predict(X_test_scaled, verbose=0).reshape(-1)
    pred = np.clip(pred, 0, None)

    mae = mean_absolute_error(y_test, pred)
    r2 = r2_score(y_test, pred)

    results = pd.DataFrame({
        "date": date_test.values,
        "actual_irrigation": y_test.values,
        "predicted_irrigation": pred,
        "error": pred - y_test.values,
    }).sort_values("date")

    print("✅ Neural network trained!")
    print(f"Hidden neurons: {hidden_neurons}")
    print(f"Epochs: {epochs}")
    print(f"Mean absolute error: {mae:.3f}")
    print(f"R² score: {r2:.3f}")

    plt.figure(figsize=(8, 4))
    plt.plot(history.history["loss"], label="training error")
    plt.plot(history.history["val_loss"], label="validation error")
    plt.xlabel("Epoch")
    plt.ylabel("Error")
    plt.title("Neural network learning curve")
    plt.legend()
    plt.show()

    plt.figure(figsize=(6, 6))
    plt.scatter(results["actual_irrigation"], results["predicted_irrigation"], alpha=0.7)
    max_val = max(results["actual_irrigation"].max(), results["predicted_irrigation"].max())
    plt.plot([0, max_val], [0, max_val], linestyle="--")
    plt.xlabel("Actual irrigation")
    plt.ylabel("Predicted irrigation")
    plt.title("Neural Network: Actual vs Predicted")
    plt.show()

    return model, scaler, results, feature_cols

nn_model, nn_scaler, nn_results, nn_features = train_neural_network(DEFAULT_FEATURES, hidden_neurons=8, epochs=80, learning_rate=0.01)
if nn_results is not None:
    display(nn_results.head())

## 🎛️ Try your own Neural Network

Click and experiment.

Questions:

- What happens with only 2 neurons?
- What happens with 32 neurons?
- What happens with 10 epochs?
- What happens with 200 epochs?
- Does the neural network always beat linear regression?

In [ ]:
if WIDGETS_AVAILABLE:
    nn_feature_picker = widgets.SelectMultiple(
        options=ALL_FEATURES,
        value=tuple(DEFAULT_FEATURES),
        description="Features",
        rows=min(12, len(ALL_FEATURES)),
        layout=widgets.Layout(width="420px")
    )
    neurons_slider = widgets.IntSlider(value=8, min=2, max=32, step=2, description="Neurons")
    epochs_slider = widgets.IntSlider(value=80, min=10, max=250, step=10, description="Epochs")
    lr_slider = widgets.FloatLogSlider(value=0.01, base=10, min=-4, max=-1, step=0.25, description="Learn rate")
    nn_button = widgets.Button(description="Train neural net", button_style="warning")
    nn_output = widgets.Output()

    def on_train_nn_clicked(_):
        with nn_output:
            clear_output()
            chosen = list(nn_feature_picker.value)
            if len(chosen) == 0:
                print("Pick at least one feature.")
                return
            global nn_model, nn_scaler, nn_results, nn_features
            nn_model, nn_scaler, nn_results, nn_features = train_neural_network(
                chosen,
                hidden_neurons=neurons_slider.value,
                epochs=epochs_slider.value,
                learning_rate=lr_slider.value,
            )
            if nn_results is not None:
                display(nn_results.head(10))

    nn_button.on_click(on_train_nn_clicked)
    display(nn_feature_picker, neurons_slider, epochs_slider, lr_slider, nn_button, nn_output)
else:
    print("Widgets are not available. Edit the numbers in the earlier cell and rerun.")

# 🏁 Final Challenge: Predict Today's Irrigation

Now the fun part.

Use today's field/weather values and ask the model:

> How much irrigation do you think happened today?

After the farmer tells us the real value, we can compare.

In [ ]:
def build_today_row(
    max_temp_f=85,
    mean_humidity_pct=40,
    precip_in=0.0,
    avg_wind_mph=8,
    et0_in=0.22,
    previous_irrigation_amount=0.2,
    days_since_irrigation=1,
):
    # Start with typical/latest values so all columns exist.
    latest = df.iloc[-1].copy()

    latest["max_temp_f"] = max_temp_f
    latest["mean_temp_f"] = max_temp_f - 12
    latest["min_temp_f"] = max_temp_f - 24
    latest["mean_humidity_pct"] = mean_humidity_pct
    latest["precip_in"] = precip_in
    latest["avg_wind_mph"] = avg_wind_mph
    latest["max_wind_mph"] = max(avg_wind_mph + 8, avg_wind_mph)
    latest["et0_in"] = et0_in
    latest["previous_irrigation_amount"] = previous_irrigation_amount
    latest["days_since_irrigation"] = days_since_irrigation

    # Simple approximations for engineered features.
    latest["temp_3day_avg_f"] = max_temp_f - 12
    latest["temp_5day_avg_f"] = max_temp_f - 13
    latest["precip_3day_sum_in"] = precip_in
    latest["precip_5day_sum_in"] = precip_in
    latest["wind_3day_avg_mph"] = avg_wind_mph
    latest["et0_3day_sum_in"] = et0_in * 3
    latest["day_of_year"] = pd.Timestamp.today().dayofyear
    latest["month"] = pd.Timestamp.today().month

    return latest


def predict_today(model_name="Linear Regression", max_temp_f=85, mean_humidity_pct=40, precip_in=0.0, avg_wind_mph=8, et0_in=0.22, previous_irrigation_amount=0.2, days_since_irrigation=1):
    if model_name == "Linear Regression":
        model = linear_model
        scaler = linear_scaler
        features = linear_features
    else:
        model = nn_model
        scaler = nn_scaler
        features = nn_features
        if model is None:
            print("Train the neural network first, or use Linear Regression.")
            return

    today = build_today_row(
        max_temp_f=max_temp_f,
        mean_humidity_pct=mean_humidity_pct,
        precip_in=precip_in,
        avg_wind_mph=avg_wind_mph,
        et0_in=et0_in,
        previous_irrigation_amount=previous_irrigation_amount,
        days_since_irrigation=days_since_irrigation,
    )

    X_today = pd.DataFrame([today[features].to_dict()])
    X_today_scaled = scaler.transform(X_today)

    if model_name == "Linear Regression":
        pred = model.predict(X_today_scaled)[0]
    else:
        pred = model.predict(X_today_scaled, verbose=0).reshape(-1)[0]

    pred = max(float(pred), 0)

    print("🌾 Today's irrigation prediction")
    print("------------------------------")
    print(f"Model: {model_name}")
    print(f"Predicted irrigation amount: {pred:.3f}")
    print("\nRemember: this is a learning model, not farm advice!")

if WIDGETS_AVAILABLE:
    model_dropdown = widgets.Dropdown(options=["Linear Regression", "Neural Network"], value="Linear Regression", description="Model")
    widgets.interact(
        predict_today,
        model_name=model_dropdown,
        max_temp_f=widgets.FloatSlider(value=85, min=50, max=110, step=1, description="Max temp F"),
        mean_humidity_pct=widgets.FloatSlider(value=40, min=10, max=100, step=1, description="Humidity %"),
        precip_in=widgets.FloatSlider(value=0.0, min=0, max=1.5, step=0.01, description="Rain in"),
        avg_wind_mph=widgets.FloatSlider(value=8, min=0, max=30, step=1, description="Wind mph"),
        et0_in=widgets.FloatSlider(value=0.22, min=0, max=0.55, step=0.01, description="ET0 in"),
        previous_irrigation_amount=widgets.FloatSlider(value=0.2, min=0, max=1.0, step=0.01, description="Prev irrig"),
        days_since_irrigation=widgets.IntSlider(value=1, min=0, max=14, step=1, description="Days since"),
    )
else:
    predict_today()

## 📝 Reflection Questions

Discuss with your group:

1. Which feature seemed most helpful?
2. Did the neural network beat linear regression?
3. Did more features always help?
4. What information might the farmer use that our model cannot see?
5. What would make this model better?

Possible answers:

- Soil moisture
- Crop type
- Crop growth stage
- Water cost
- Equipment problems
- Farmer experience
- Weather forecast, not just past weather

## ⚠️ Important

This notebook is for learning.

Real irrigation decisions should be made by farmers, agronomists, irrigation specialists, and appropriate farm systems.

Machine learning can help us ask better questions, but it does not replace real-world expertise.